[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/12_embedders_modernos/12_embedders_modernos.ipynb)

# Módulo 12 · Embedders modernos: instrucciones, Matryoshka y cuantización

**Nivel:** 🟠 Avanzado · **Duración:** ≈ 5 h en total (lección ≈ 3 h + proyecto ≈ 2 h) · **GPU recomendada:** T4 (todo corre en CPU con `FAST_DEV_RUN = True`) ·
**Cómputo estimado:** 0 unidades en CPU; ≈ 1–3 unidades de Colab en T4 con `FAST_DEV_RUN = False` (estimación orientativa: depende de cuántos modelos actives).

**Prerrequisitos:** módulo 09 (matriz de embeddings y geometría), módulo 10 (pooling, anisotropía, similitudes) y módulo 11 (InfoNCE, hard negatives, MRL como "ingrediente" contrastivo).

**Hilo conductor — Políglota.** Eres ML engineer en *Políglota Labs*. Ya tienes un baseline BM25 (00) y entrenaste tu propio embedder (11). Ahora el CTO pregunta:
*«¿Qué embedder comprobado ponemos en producción, a qué dimensión y con qué cuantización, para que 10 M de documentos quepan en un nodo?»*
Este módulo te da el criterio (y el código) para contestar con datos, no con el leaderboard.

> **Estado del arte a octubre de 2026.** Este campo cambia cada pocas semanas: toda cifra, ID de modelo o precio va **fechado** y debe re-verificarse en la tarjeta del modelo antes de decidir. Lo que no pude verificar no aparece como cifra.

## 🎯 Objetivos de aprendizaje
Al terminar el módulo serás capaz de:
1. **Explicar** las tres olas de embedders (encoders contrastivos → LLM-as-embedder → modelos multi-tarea con Matryoshka/QAT) y el rol de cada ingrediente de la receta.
2. **Implementar desde cero** los pooling CLS / mean / max / **last-token** / **latent attention** (NV-Embed) con máscaras de padding correctas, y detectar el bug de *padding side*.
3. **Aplicar** prefijos e instrucciones de query/documento de cada familia (E5, Qwen3-Embedding, EmbeddingGemma, Nomic…) y **medir** su impacto.
4. **Derivar e implementar** la pérdida Matryoshka (MRL), el truncado + renormalización, y **demostrar** con un experimento por qué truncar un modelo *no* entrenado con MRL falla.
5. **Implementar desde cero** cuantización escalar `int8` (con calibración por dimensión) y **binaria** (bits empaquetados + distancia de Hamming + *rescoring*), y compararla con `sentence-transformers`.
6. **Comparar** embedders reales con `sentence-transformers` en calidad (nDCG@10), memoria y latencia, con **intervalos de confianza** por bootstrap.
7. **Leer críticamente** MTEB/MMTEB: qué miden, qué no, y por qué tu golden set manda.
8. **Decidir** con un árbol de decisión explícito qué embedder, dimensión y precisión usar para un caso de negocio.

## 🔁 Conexión con módulos anteriores
Antes de empezar, intenta contestar de memoria (la respuesta está plegada):

<details><summary><b>1 · (M11)</b> ¿Qué hace InfoNCE con los negativos <i>in-batch</i> y qué controla la temperatura $\tau$?</summary>

Trata a los otros $B-1$ documentos del lote como negativos de cada query y maximiza la probabilidad softmax del positivo: $\mathcal{L}=-\log\frac{e^{s(q,d^+)/\tau}}{\sum_j e^{s(q,d_j)/\tau}}$. Una $\tau$ pequeña (≈ 0.02–0.05) afila la distribución y da más peso a los negativos difíciles. Hoy ese mismo bloque de código se *reusa* para Matryoshka: se calcula varias veces sobre prefijos del vector.
</details>

<details><summary><b>2 · (M10)</b> ¿Por qué el <i>mean pooling</i> suele ganar al CLS en modelos sin fine-tuning contrastivo, y qué es la anisotropía?</summary>

El estado del `[CLS]` solo se entrena para la tarea de pre-entrenamiento (NSP/MLM no lo exigen como resumen), mientras que el promedio agrega información de todos los tokens. La anisotropía es que los embeddings ocupan un cono estrecho del espacio (cosenos altos entre textos no relacionados); el entrenamiento contrastivo la corrige.
</details>

<details><summary><b>3 · (M09)</b> ¿Qué dimensión tiene la capa de embedding de tokens de un LLM pequeño y qué relación tiene con la dimensión de salida del embedder de oraciones?</summary>

La dimensión oculta $d_{model}$ (p. ej. 1024 en un decoder de ≈ 0.6 B). Un embedder basado en ese LLM suele exponer $d_{model}$ como dimensión del vector (y Matryoshka permite usar prefijos más cortos).
</details>

<details><summary><b>4 · (M04)</b> ¿Por qué la tokenización influye en el "contexto efectivo" de un embedder en español vs inglés?</summary>

Porque la <i>fertilidad</i> (tokens por palabra) del español suele ser mayor que la del inglés en vocabularios sesgados al inglés: el mismo texto ocupa más tokens y se trunca antes en `max_seq_length`. Por eso Políglota fija `max_len` en tokens, no en caracteres.
</details>

## 🗺️ Mapa de madurez (a oct-2026)
🟢 sólido (estándar de industria) · 🟡 emergente (resultados publicados, adopción parcial) · 🔴 experimental.

| Técnica | Madurez | Quién la usa (respaldado) | Fuente |
|---|---|---|---|
| Encoders contrastivos multilingües pequeños/medianos (E5, BGE-M3, GTE) | 🟢 | Microsoft (E5), BAAI (BGE-M3), Alibaba (mGTE) | Wang et al. 2024 (2402.05672); Chen et al. 2024 (2402.03216); Zhang et al. 2024 (2407.19669) |
| Prefijos / instrucciones de query y documento | 🟢 | E5 (`query:`/`passage:`), Qwen3-Embedding (`Instruct:`), EmbeddingGemma (prompts por tarea) | tarjetas de modelo; Wang et al. 2023 (2401.00368) |
| **Matryoshka (MRL)**: truncar + renormalizar | 🟢 | OpenAI (`dimensions`), Cohere Embed v4, Nomic v2, EmbeddingGemma, Qwen3-Embedding, Gemini Embedding, Voyage | Kusupati et al. 2022 (2205.13147); docs de cada proveedor |
| Cuantización `int8` de embeddings | 🟢 | `sentence-transformers`, Voyage (salida int8), Cohere | HF blog (Shakir, Aarsen, Lee 2024); docs |
| Cuantización **binaria** + rescoring | 🟡 | Cohere/Voyage ofrecen salida binaria; Elastic BBQ (vía RaBitQ, módulo 14). La calidad depende del modelo | HF blog 2024; Elastic Search Labs |
| *Quantization-aware training* de embedders | 🟡 | Voyage-4 (según su documentación en MongoDB/Azure) | docs de Voyage |
| LLM-as-embedder 0.5–1 B (Qwen3-Embedding-0.6B, Jina v5) | 🟢 | Alibaba, Jina | 2506.05176; página del modelo |
| LLM-as-embedder 7–8 B (e5-mistral, NV-Embed, Qwen3-Embedding-8B) | 🟡 | NVIDIA, Microsoft, Alibaba — coste de inferencia alto | 2401.00368; 2405.17428; 2506.05176 |
| Pooling *latent attention* (NV-Embed) | 🟡 | NVIDIA | 2405.17428 |
| LLM2Vec (decoder → encoder bidireccional) | 🟡 | investigación (McGill/Mila) | 2404.05961 |
| Embeddings multimodales unificados (Gemini Embedding 2, Jina v4/v5-omni) | 🟡 | Google (GA abr-2026), Jina — recientes; tema del módulo 16 | blog de Google (mar/abr-2026); Elastic (may-2026) |
| MoE para embeddings (Nomic Embed v2) | 🟡 | Nomic | 2502.07972 |
| Elegir embedder **solo** por el promedio de MTEB | 🔴 (mala práctica) | — | ver sección 7 |
| Evaluar en tu propio golden set con intervalos de confianza | 🟢 | práctica estándar en equipos de search | módulo 15 |

## 0 · Preparación del entorno
Las celdas de utilidades (corpus, métricas, embedder) están en esta lección porque **cada notebook del curso es autocontenido**.
Si no hay acceso a Hugging Face (proxy, sin red), todo sigue funcionando con un **fallback** que te avisa con un `⚠️`: los números cambian, los conceptos no.

In [ ]:
!pip install -q -U sentence-transformers transformers scikit-learn matplotlib pandas mteb

In [ ]:
import os, sys, time, math, random, json, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU en minutos). Pon False en Colab con GPU para la escala completa.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))

try:
    import torch
    torch.manual_seed(SEED)
    torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:      # los fragmentos "desde cero" de NumPy funcionan sin torch
    torch, device = None, "cpu"
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

### Corpus Políglota y métricas
`carga_corpus(fast_dev)` sigue el **contrato común del curso**: devuelve `{"docs", "queries", "qrels", "source"}` con cadena *oficial → espejo → sintético*. Si ves `source = "synthetic"`, estás usando el corpus **sintético** del módulo 00 (texto ES/EN generado con plantillas + funciones reales de la biblioteca estándar de Python como "código"): el mismo esquema que el real, pero **los números no son comparables** con los del corpus real. Es la implementación **canónica** de `carga_corpus` escrita en el módulo 00 (cadena oficial → espejo → sintético).

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
def ndcgs_por_query(run: dict, qrels: dict, k: int = 10):
    """nDCG@k de cada query (mismo filtro que evaluate(): solo queries con algún relevante). Sirve para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), r, k) for q, r in qrels.items() if any(v > 0 for v in r.values())])

### Un envoltorio común para embedders
`Embedder` aplica el prefijo correcto de cada familia y siempre devuelve vectores `float32` normalizados. Si el modelo no se puede descargar, `cargar_embedder` devuelve un **fallback TF-IDF+SVD** (no semántico ni multilingüe): el pipeline sigue ejecutándose para que puedas practicar truncado, cuantización y métricas.

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
CORPUS = carga_corpus(FAST_DEV_RUN)
DOC_IDS = [d["id"] for d in CORPUS["docs"]]
DOC_TXT = [d["text"] for d in CORPUS["docs"]]
QRYS = CORPUS["queries"]
QRELS = CORPUS["qrels"]
print(CORPUS["source"], len(DOC_IDS), "docs ·", len(QRYS), "queries")

def run_denso(Eq, Ed, k=10):
    """Búsqueda exacta por producto punto (embeddings normalizados -> coseno)."""
    S = Eq @ Ed.T
    return {q["qid"]: rank_from_scores(S[i], DOC_IDS, k) for i, q in enumerate(QRYS)}

# Embedder "principal" de la lección: el primero que se pueda cargar de verdad; si no, el fallback.
emb = None
for clave in ("multilingual-e5-small", "paraphrase-multilingual-MiniLM-L12-v2"):
    emb = cargar_embedder(clave)
    if emb is not None:
        break
if emb is None:
    emb = cargar_embedder("multilingual-e5-small", textos_ajuste=DOC_TXT)
t0 = time.perf_counter()
E_d = emb.encode_docs(DOC_TXT)
E_q = emb.encode_queries([q["text"] for q in QRYS])
print(f"embedder = {emb.clave} · real={emb.is_real} · dim={E_d.shape[1]} · {time.perf_counter()-t0:.1f}s")
base = evaluate(run_denso(E_q, E_d), QRELS)
print({k: round(float(v), 3) for k, v in base.items()})

## 1 · Intuición: tres olas de embedders

**Ola 1 (2019–2022) — encoders contrastivos pequeños.** Sentence-BERT, luego E5/BGE/GTE: un encoder bidireccional (BERT/XLM-R) con *mean pooling*, entrenado primero con una enorme cantidad de pares débilmente supervisados y luego con datos etiquetados y *hard negatives*. 100–500 M de parámetros, 384–1024 dimensiones, 512 tokens.

**Ola 2 (2023–2024) — LLM-as-embedder.** Se observa que un decoder grande (Mistral-7B) fine-tuneado con pérdida contrastiva supera a los encoders: e5-mistral (datos sintéticos generados por LLM), NV-Embed (pooling *latent attention*, máscara causal retirada), LLM2Vec (convertir un decoder en encoder bidireccional). Coste: un decoder de 7 B tiene ≈ 20× más parámetros que un encoder de 300 M, y eso se paga en cada texto que indexas.

**Ola 3 (2025–2026) — modelos "de producto".** Familias con **varios tamaños**, **contexto largo (8k–32k)**, **instrucciones**, **Matryoshka** y a veces **cuantización consciente** (QAT), multilingües y, cada vez más, **multimodales** (módulo 16): Qwen3-Embedding (0.6B/4B/8B), Gemini Embedding, EmbeddingGemma (≈ 300 M), Jina v4/v5, Nomic v2 (MoE), Voyage-4, Cohere Embed v4.

> 💡 **Analogía con compresión y recsys.** Un embedder es un *codec con pérdida* diseñado para que la **distancia** (no la reconstrucción) se preserve. Matryoshka es un codec *escalable* (como JPEG progresivo: los primeros bytes ya dan una imagen útil) y la cuantización es el *segundo* codec encima. En recsys, la torre del ítem hace el mismo papel: producir un vector indexable (módulo `recsys-course/08`).

> ### 🎨 GEMINI-IMAGEN · ID: `M12-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **La receta de un embedder moderno**
>
> **Qué muestra:** Pipeline de entrenamiento de un embedder de 2025-2026 (backbone → pretraining contrastivo → fine-tuning con instrucciones y hard negatives → Matryoshka/QAT → vector) y las dos variantes de backbone.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Pipeline de entrenamiento de un embedder moderno de izquierda a derecha con cinco etapas
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9, estilo de infografía de ingeniería. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif legible. Sin ilustraciones decorativas ni texto inventado.
> 
> Composición de IZQUIERDA a DERECHA con 5 cajas redondeadas conectadas por flechas grises gruesas:
> 1) Caja gris con título "Backbone". Dentro, dos mini-opciones apiladas: "Encoder bidireccional (XLM-R, ModernBERT)" en azul y "Decoder LLM (Qwen3, Gemma, Mistral)" en naranja.
> 2) Caja azul: "Pretraining contrastivo" con subtítulo "pares débilmente supervisados · InfoNCE · in-batch negatives".
> 3) Caja azul: "Fine-tuning supervisado" con subtítulo "hard negatives · instrucciones de tarea · multilingüe".
> 4) Caja verde: "Matryoshka + QAT (opcional)" con subtítulo "pérdida sobre prefijos del vector · cuantización consciente".
> 5) Caja gris final: "Vector de 1024 dimensiones" dibujado como una barra horizontal segmentada en bloques; los primeros 4 bloques resaltados en verde con la etiqueta "prefijo útil (Matryoshka)".
> 
> Debajo de las cajas 1 y 2, una fina línea punteada naranja con la etiqueta "pooling: mean (encoder) / last-token o latent attention (decoder)".
> En la parte inferior, un pie de figura pequeño: "Cada flecha = una etapa de entrenamiento".
> 
> Evita: texto ilegible o diminuto, flechas ambiguas o cruzadas, logos de empresas, iconos decorativos, números de parámetros inventados.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M12-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Pipeline de entrenamiento de un embedder moderno de izquierda a derecha con cinco etapas](assets/M12-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> [Backbone: encoder | decoder] -> [Pretraining contrastivo] -> [Fine-tuning: hard negatives + instrucciones]
>                              -> [Matryoshka + QAT] -> [vector 1024-d: ████ prefijo útil | ░░░░░░░░░░]
> pooling: mean (encoder) / last-token o latent attention (decoder)
> ```

> 👀 **Qué debes observar.** La diferencia entre familias casi nunca está en la *arquitectura* sino en las **dos últimas cajas**: la calidad y variedad de los datos de fine-tuning y la forma de empaquetar el vector (Matryoshka, cuantización). Eso explica por qué dos modelos del mismo tamaño pueden diferir 5–10 puntos en tu dominio.

## 2 · Pooling en encoders y en decoders (desde cero)

Un transformer produce una matriz de estados ocultos $H\in\mathbb{R}^{T\times d}$ por texto. El embedder necesita **un** vector $e\in\mathbb{R}^d$:

| Pooling | Fórmula | Quién lo usa |
|---|---|---|
| CLS | $e = h_0$ | BGE (denso), muchos encoders |
| Mean | $e = \frac{\sum_t m_t h_t}{\sum_t m_t}$ | E5, SBERT, Nomic |
| Max | $e_j = \max_t h_{t,j}$ | histórico |
| **Last-token** | $e = h_{\ell-1}$ (último token **real**, con $\ell=\sum_t m_t$) | e5-mistral, Qwen3-Embedding (según sus papers/tarjetas) |
| **Latent attention** | $e = \text{mean}_t\big(\text{MLP}(\text{softmax}(HL^\top/\sqrt d)\,L)\big)$ con diccionario entrenable $L\in\mathbb{R}^{r\times d}$ | NV-Embed |

$m_t\in\{0,1\}$ es la **máscara de atención** (1 = token real). En un decoder con atención causal el único estado que "ha visto" todo el texto es el **último**, por eso se usa last-token (típicamente tras añadir un token de fin de secuencia).

> ⚠️ **El bug clásico del last-token.** Con *padding a la derecha*, el último índice del tensor es un token de relleno; hay que tomar el índice $\ell-1$ por fila. Con *padding a la izquierda* el último índice sí es el correcto. Mezclar ambos convenios degrada la calidad **sin dar error**.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))

def pool_cls(H, m):  return H[:, 0]
def pool_mean(H, m):
    w = m[..., None].astype(H.dtype)
    return (H * w).sum(1) / np.maximum(w.sum(1), 1)
def pool_max(H, m):
    return np.where(m[..., None] == 1, H, -np.inf).max(1)
def pool_last(H, m, padding_side="right"):
    if padding_side == "left":                 # el último índice del tensor ya es el último token real
        return H[:, -1]
    idx = m.sum(1) - 1                         # padding a la derecha: índice del último token real por fila
    return H[np.arange(len(H)), idx]
def pool_latent(H, m, L, W1, W2):
    """Latent attention de NV-Embed: los estados H consultan un diccionario entrenable L (r x d); luego MLP y media."""
    att = softmax(H @ L.T / np.sqrt(H.shape[-1]))        # (B,T,r)
    O = att @ L                                          # (B,T,d)
    O = O + gelu(O @ W1) @ W2                            # MLP residual
    return pool_mean(O, m)

rng = np.random.default_rng(SEED)
B, T, D = 2, 6, 8
H = rng.normal(size=(B, T, D))
m_right = np.array([[1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1]])   # fila 0 tiene 2 tokens de relleno a la derecha
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

correcto = pool_last(H, m_right, "right")[0]
ingenuo = H[0, -1]                                             # el bug: tomar siempre el último índice
print("last-token correcto  = estado del token 3:", np.allclose(correcto, H[0, 3]))
print("last-token ingenuo   = estado del PAD   :", np.allclose(ingenuo, H[0, 5]), "· coseno con el correcto:", round(cos(correcto, ingenuo), 3))

L = rng.normal(size=(4, D)); W1 = rng.normal(size=(D, 2 * D)) * 0.2; W2 = rng.normal(size=(2 * D, D)) * 0.2
for nombre, f in [("cls", pool_cls), ("mean", pool_mean), ("max", pool_max), ("last", pool_last)]:
    print(f"{nombre:>5} ->", np.round(f(H, m_right)[0][:4], 2))
print("latent ->", np.round(pool_latent(H, m_right, L, W1, W2)[0][:4], 2))

In [ ]:
import matplotlib.pyplot as plt

T = 6
causal = np.tril(np.ones((T, T)))
bidir = np.ones((T, T))
pesos = {                                   # peso de cada token en el vector final (texto de 6 tokens, sin padding)
    "CLS": np.eye(T)[0], "Mean": np.ones(T) / T, "Last-token": np.eye(T)[-1],
}
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1, 1, 1.25]})
for ax, mat, tit in ((axs[0], bidir, "Encoder: atención bidireccional"), (axs[1], causal, "Decoder: atención causal")):
    ax.imshow(mat, cmap="Blues", vmin=0, vmax=1.4)
    ax.set_title(tit, fontsize=10); ax.set_xlabel("token atendido (clave)"); ax.set_ylabel("token que atiende (query)")
    ax.set_xticks(range(T)); ax.set_yticks(range(T))
im = axs[2].imshow(np.stack(list(pesos.values())), cmap="Oranges", vmin=0, vmax=1, aspect="auto")
axs[2].set_yticks(range(len(pesos))); axs[2].set_yticklabels(list(pesos)); axs[2].set_xticks(range(T))
axs[2].set_xlabel("posición del token"); axs[2].set_title("Peso de cada token en el embedding", fontsize=10)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) En el decoder (triángulo) solo la **última fila** ve a todos los tokens: por eso last-token es el único estado "completo". (2) Mean pooling reparte peso uniforme, CLS y last-token dependen de **un solo** estado. (3) Esto explica por qué LLM2Vec y NV-Embed **quitan la máscara causal** al entrenar: con atención bidireccional, hasta el mean pooling pasa a ser razonable en un decoder.

### LLM2Vec y NV-Embed en dos párrafos
- **LLM2Vec** (BehnamGhader et al., 2024) convierte *cualquier* decoder en encoder con 3 pasos baratos: (1) **activar atención bidireccional**, (2) **MNTP** (*masked next token prediction*), (3) **SimCSE no supervisado**. Es 🟡: mejora mucho al decoder "crudo", pero un modelo entrenado desde el principio para embeddings (Qwen3-Embedding, e5-mistral) suele ser más sólido.
- **NV-Embed** (Lee et al., 2024) aplica el pooling *latent attention* de la tabla, **elimina la máscara causal** durante el entrenamiento contrastivo y usa un fine-tuning contrastivo en **dos etapas** (primero con instrucciones y negativos in-batch para retrieval; luego mezcla tareas no-retrieval).
- **e5-mistral** (Wang et al., 2023/24) genera datos sintéticos con un LLM propietario para muchísimas tareas e idiomas y fine-tunea Mistral-7B con pérdida contrastiva; el embedding es el estado oculto del **token EOS final** (last-token).

Todo esto se resume en: *el backbone importa menos que los datos y que la forma de empaquetar el vector*.

## 3 · Embeddings con instrucciones y prefijos

Un embedder moderno es **asimétrico**: la *query* ("¿cómo recorto un vector?") y el *documento* (un párrafo que lo explica) son textos de naturaleza distinta. La solución estándar es **anteponer un prefijo** distinto a cada lado:

| Familia | Query | Documento | Fuente |
|---|---|---|---|
| E5 (multilingual-e5) | `query: …` | `passage: …` | tarjeta del modelo |
| Nomic Embed | `search_query: …` | `search_document: …` | tarjeta del modelo |
| Qwen3-Embedding | `Instruct: {tarea}\nQuery:{texto}` | sin prefijo | tarjeta del modelo |
| EmbeddingGemma | `task: search result \| query: …` | `title: none \| text: …` | tarjeta / blog de HF |
| BGE-M3, GTE-multilingual | sin prefijo | sin prefijo | tarjetas |

Con `sentence-transformers` ≥ 5 los métodos `encode_query()` / `encode_document()` aplican automáticamente los prompts que el autor guardó en la configuración del modelo (si existen). Aquí usamos prefijos **explícitos** para ver qué pasa.

> 🧪 **Experimento.** Mide el mismo embedder con (a) prefijos correctos, (b) sin prefijos y (c) prefijos **invertidos**. Con un modelo entrenado con prefijos (E5, Qwen3, EmbeddingGemma…), (b) y (c) dañan la calidad; con modelos sin prefijos no pasa nada. Si estás en el *fallback* TF-IDF, el prefijo es solo ruido de texto: el experimento sigue ejecutándose pero no representa al modelo real.

In [ ]:
def ndcg_con_prefijos(emb, prefijo_q, prefijo_d):
    Eq = emb._enc([q["text"] for q in QRYS], prefijo_q)
    Ed = emb._enc(DOC_TXT, prefijo_d)
    return evaluate(run_denso(Eq, Ed), QRELS)["ndcg"]

cfg = emb.cfg
experimentos = {
    "prefijos correctos": (cfg.get("q", ""), cfg.get("d", "")),
    "sin prefijos": ("", ""),
    "invertidos": (cfg.get("d", ""), cfg.get("q", "")),
}
res_prefijos = {nombre: ndcg_con_prefijos(emb, *pq) for nombre, pq in experimentos.items()}
for k, v in res_prefijos.items():
    print(f"{k:>20}: nDCG@10 = {v:.3f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.4, 3.2))
colores = ["#2CA02C", "#6B7280", "#F28E2B"]
ax.bar(list(res_prefijos), list(res_prefijos.values()), color=colores)
ax.set_ylabel("nDCG@10"); ax.set_title(f"Efecto del prefijo en {emb.clave}")
ax.set_ylim(0, max(res_prefijos.values()) * 1.25)
for i, v in enumerate(res_prefijos.values()):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con un modelo de la familia E5 real, la barra verde (prefijos correctos) suele ser la más alta y los prefijos invertidos pierden calidad; el tamaño de la caída depende del modelo y del dominio, **mídelo con tu golden set** en vez de asumirlo. Con modelos sin prefijos las tres barras son casi iguales. El error silencioso más común en producción es **olvidar el prefijo del lado de la query o del documento** (ver «Errores comunes»).

## 4 · Matryoshka (MRL): vectores que se pueden recortar

**Problema.** Un embedder de 1024 dimensiones cuesta 4 KB por documento en `float32`. Si solo necesitas 256 dimensiones, ¿puedes quedarte con las *primeras* 256? Con un modelo normal, **no**: la información está repartida por igual en todas las coordenadas (cualquier rotación ortogonal del espacio es equivalente para el coseno).

**Idea (Kusupati et al., 2022).** Entrenar de modo que **cada prefijo** $z_{1:m}$ ($m\in\mathcal{M}$, típicamente potencias de 2) sea por sí solo un buen embedding. Sea $z=f_\theta(x)\in\mathbb{R}^{d}$ y $\mathcal{M}=\{m_1<\dots<m_K=d\}$:

$$\mathcal{L}_{MRL}=\sum_{m\in\mathcal{M}} c_m\;\mathcal{L}_{InfoNCE}\!\left(\frac{z^q_{1:m}}{\lVert z^q_{1:m}\rVert},\frac{z^d_{1:m}}{\lVert z^d_{1:m}\rVert}\right)$$

- $c_m$: peso de cada escala (en la práctica 1 para todas, o más peso a las pequeñas).
- La **renormalización** de cada prefijo dentro de la pérdida es esencial: es la misma operación que harás al truncar en inferencia.
- Coste de entrenamiento: $K$ evaluaciones de una matriz $B\times B$ de similitudes (despreciable frente al backbone).

**En inferencia:** `e_m = e[:, :m] / ||e[:, :m]||`. Sin la renormalización, el producto punto mezcla normas distintas y el ranking cambia.

**Cuándo NO usarlo:** si tu modelo *no* fue entrenado con MRL, truncar degrada rápido; en ese caso usa PCA (aprende una base ordenada por varianza) o un modelo con MRL. Lo comprobamos a continuación.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def toy_datos(n, d_in=128, d_lat=24, ruido=0.35, seed=0):
    """Dos 'vistas' ruidosas del mismo vector latente: el par positivo del aprendizaje contrastivo."""
    g = torch.Generator().manual_seed(seed)
    A = torch.randn(d_lat, d_in, generator=g) / math.sqrt(d_lat)
    Z = torch.randn(n, d_lat, generator=g)
    vista = lambda: Z @ A + ruido * torch.randn(n, d_in, generator=g)
    return vista(), vista()

def info_nce(a, b, tau=0.05):
    a, b = F.normalize(a, dim=-1), F.normalize(b, dim=-1)
    logits = a @ b.T / tau
    y = torch.arange(len(a))
    return 0.5 * (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y))

def entrena_toy(dims_mrl, d_out=64, pasos=300, bs=256, seed=0):
    torch.manual_seed(seed)
    enc = nn.Sequential(nn.Linear(128, 256), nn.GELU(), nn.Linear(256, d_out))
    opt = torch.optim.Adam(enc.parameters(), lr=2e-3)
    x1, x2 = toy_datos(8000, seed=1)
    for _ in range(pasos):
        idx = torch.randint(0, len(x1), (bs,))
        a, b = enc(x1[idx]), enc(x2[idx])
        loss = sum(info_nce(a[:, :m], b[:, :m]) for m in dims_mrl) / len(dims_mrl)   # <- la pérdida MRL
        opt.zero_grad(); loss.backward(); opt.step()
    return enc

@torch.no_grad()
def recall1_por_dim(enc, dims, n=2000):
    t1, t2 = toy_datos(n, seed=7)
    a, b = enc(t1), enc(t2)
    out = {}
    for m in dims:
        s = F.normalize(a[:, :m], dim=-1) @ F.normalize(b[:, :m], dim=-1).T
        out[m] = (s.argmax(1) == torch.arange(n)).float().mean().item()
    return out

DIMS_EVAL = [2, 4, 8, 16, 32, 48, 64]
enc_std = entrena_toy([64])                 # entrenamiento estándar: solo la dimensión completa
enc_mrl = entrena_toy([8, 16, 32, 64])      # Matryoshka: pérdida en 4 prefijos
r_std, r_mrl = recall1_por_dim(enc_std, DIMS_EVAL), recall1_por_dim(enc_mrl, DIMS_EVAL)
print("dim   estándar   MRL")
for m in DIMS_EVAL:
    print(f"{m:>3}   {r_std[m]:.3f}      {r_mrl[m]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.plot(DIMS_EVAL, [r_std[m] for m in DIMS_EVAL], "o-", color="#F28E2B", label="entrenamiento estándar")
ax.plot(DIMS_EVAL, [r_mrl[m] for m in DIMS_EVAL], "s-", color="#1F6FEB", label="Matryoshka (dims 8/16/32/64)")
ax.set_xscale("log", base=2); ax.set_xticks(DIMS_EVAL); ax.set_xticklabels(DIMS_EVAL)
ax.set_xlabel("dimensiones conservadas (prefijo renormalizado)"); ax.set_ylabel("Recall@1 (retrieval del par)")
ax.set_title("Juguete: truncar un embedder con y sin pérdida Matryoshka"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** A 64 dimensiones ambos modelos son prácticamente iguales (MRL **no** cuesta calidad a plena dimensión). A 8–16 dimensiones, el modelo estándar se desploma mientras el Matryoshka conserva buena parte del recall: la información "importante" quedó concentrada en las primeras coordenadas. Fuera de las escalas entrenadas (p. ej. 48) la curva MRL sigue siendo suave, pero lo seguro es truncar **solo** a dimensiones soportadas por el proveedor.

> ### 🎨 GEMINI-IMAGEN · ID: `M12-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Matryoshka: el vector como muñecas rusas**
>
> **Qué muestra:** Un vector de 1024 dimensiones con sus prefijos de 512, 256, 128 y 64 dimensiones anidados, el paso de renormalización y el coste en memoria de cada escala.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cinco barras anidadas de 1024 a 64 dimensiones con su coste en bytes y el paso de renormalización
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif legible. Sin ilustraciones decorativas.
> 
> Dibuja CINCO barras horizontales anidadas alineadas a la izquierda (como muñecas rusas), de arriba abajo, cada una más corta que la anterior, todas con el mismo alto:
> - Barra gris muy larga con la etiqueta "1024 dims · 4096 bytes (float32)".
> - Barra azul a la mitad de longitud: "512 dims · 2048 bytes".
> - Barra azul más corta: "256 dims · 1024 bytes".
> - Barra azul aún más corta: "128 dims · 512 bytes".
> - Barra verde la más corta: "64 dims · 256 bytes".
> Los primeros 64 bloques de todas las barras comparten el mismo color verde para mostrar que el prefijo es común, con una llave vertical a la izquierda que dice "mismo prefijo".
> 
> A la derecha de la barra de 256 dims, una flecha naranja hacia un cuadro con el texto "renormalizar: e / ||e||" y debajo una circunferencia pequeña (vector unitario) con la leyenda "||e|| = 1".
> Arriba a la izquierda, el título: "Matryoshka: cada prefijo es un embedding útil".
> Abajo, una nota en gris: "Recortar + renormalizar = ahorrar memoria sin reentrenar (solo si el modelo se entrenó con MRL)".
> 
> Evita: añadir más barras o números distintos a los indicados, texto ilegible, efectos 3D, imágenes de muñecas reales, logos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M12-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Cinco barras anidadas de 1024 a 64 dimensiones con su coste en bytes y el paso de renormalización](assets/M12-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> 1024 dims  ████████████████████████████████  4096 B
>  512 dims  ████████████████                  2048 B
>  256 dims  ████████                          1024 B   -> renormalizar e/||e||
>  128 dims  ████                               512 B
>   64 dims  ██                                 256 B   (prefijo común)
> ```

> 👀 **Qué debes observar.** La memoria baja **linealmente** con la dimensión (÷2 por cada escala), mientras que, en un modelo con MRL, la calidad cae mucho más despacio. Esa asimetría es lo que hace rentable el truncado: pasar de 1024 a 256 dimensiones ahorra 4× de RAM por una pérdida de calidad que *debes medir* en tu corpus.

> ### 🎬 GEMINI-VIDEO · ID: `M12-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Truncar y renormalizar paso a paso**
>
> **Qué muestra:** Animación de un embedding que se recorta a un prefijo, se renormaliza y se usa para ordenar 5 documentos; contrasta un modelo con Matryoshka y otro sin él.
> **Formato:** MP4 16:9, 1080p | Duración: 25-30 s | **Texto alternativo:** Animación de truncado y renormalización de un embedding con dos rankings comparados
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video explicativo de 16:9, fondo blanco, estilo diagrama técnico limpio, paleta azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280, tipografía sans-serif, ritmo calmado, SIN voz en off ni música; solo texto en pantalla en español (términos técnicos en inglés).
> 
> Escena 1 (0–5 s): aparece una barra horizontal gris dividida en 16 bloques con la etiqueta "embedding completo (d = 16, ejemplo)". Título arriba: "Truncar un embedding".
> Escena 2 (5–10 s): los 8 primeros bloques se vuelven azules y los otros 8 se desvanecen; etiqueta "m = 8: nos quedamos con el prefijo". Un contador de memoria cambia de "64 bytes" a "32 bytes".
> Escena 3 (10–16 s): una flecha naranja vector 2D (dibujada como flecha desde el origen) crece o se encoge hasta tocar un círculo unitario punteado; etiqueta "renormalizar: e / ||e||". La flecha termina con longitud 1.
> Escena 4 (16–24 s): se muestran 5 documentos D1…D5 como puntos en el círculo y una query Q como flecha verde; una lista a la derecha ordena los documentos por similitud coseno. Con "modelo Matryoshka" el orden es D3 > D1 > D4 > D2 > D5 (igual que con el vector completo); con "modelo estándar truncado" el orden cambia a D2 > D5 > D3 > D1 > D4 y se marca en naranja "ranking roto".
> Escena 5 (24–28 s): fundido con el texto "Matryoshka: el prefijo ya es un buen embedding. Sin MRL, el truncado destruye el ranking."
> 
> Evita: voz, música, texto ilegible, números distintos a los dados, elementos decorativos, cambios bruscos de cámara.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M12-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M12-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> [d=16] ████████████████  64 B
> [m=8 ] ████████          32 B  -> renormalizar (||e||=1)
> Matryoshka : D3 > D1 > D4 > D2 > D5  (ranking conservado)
> Estándar   : D2 > D5 > D3 > D1 > D4  (ranking roto)
> ```

> 👀 **Qué debes observar.** La renormalización devuelve el vector a la esfera unidad (el coseno vuelve a ser un producto punto) y, con un modelo Matryoshka, el **orden** de los documentos no cambia. Con un modelo no entrenado así, el ranking se rompe.

### Truncado en un embedder real (o fallback)
Medimos nDCG@10 al truncar `E_d`/`E_q` a distintas dimensiones con tres estrategias: **prefijo + renormalizar** (lo correcto para MRL), **prefijo sin renormalizar** (el error típico) y **PCA** (la alternativa cuando el modelo *no* es Matryoshka).

In [ ]:
def trunc_renorm(E, m):
    X = E[:, :m]
    return (X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)).astype(np.float32)

def pca_fit(Ed, m):
    mu = Ed.mean(0, keepdims=True)
    _, _, Vt = np.linalg.svd(Ed - mu, full_matrices=False)
    return mu, Vt[:m]

def ndcg_de(Eq, Ed):
    return evaluate(run_denso(Eq, Ed), QRELS)["ndcg"]

D = E_d.shape[1]
dims = sorted({m for m in (8, 16, 32, 64, 128, 256, 384, 512, 768, 1024) if m < D} | {D})
curvas = {"prefijo + renormalizar": [], "prefijo sin renormalizar (dot)": [], "PCA + renormalizar": []}
for m in dims:
    curvas["prefijo + renormalizar"].append(ndcg_de(trunc_renorm(E_q, m), trunc_renorm(E_d, m)))
    curvas["prefijo sin renormalizar (dot)"].append(ndcg_de(E_q[:, :m], E_d[:, :m]))
    mu, W = pca_fit(E_d, m)
    proy = lambda E: trunc_renorm((E - mu) @ W.T, W.shape[0])
    curvas["PCA + renormalizar"].append(ndcg_de(proy(E_q), proy(E_d)))

fig, ax = plt.subplots(figsize=(7, 3.8))
for (nombre, ys), c in zip(curvas.items(), ["#1F6FEB", "#F28E2B", "#2CA02C"]):
    ax.plot(dims, ys, "o-", label=nombre, color=c)
ax.set_xscale("log", base=2); ax.set_xticks(dims); ax.set_xticklabels(dims, rotation=45)
ax.set_xlabel("dimensiones conservadas"); ax.set_ylabel("nDCG@10")
ax.set_title(f"Truncado de {emb.clave} en Políglota ({CORPUS['source']})"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print({m: round(float(v), 3) for m, v in zip(dims, curvas["prefijo + renormalizar"])})

> 👀 **Qué debes observar.** (1) Si el modelo es **Matryoshka** (Nomic v2, Qwen3-Embedding, EmbeddingGemma…), "prefijo + renormalizar" cae despacio. (2) Si **no** lo es (p. ej. `multilingual-e5-small`, el modelo por defecto de esta celda, o el fallback), la curva del prefijo cae antes y **PCA** la supera: ordena las direcciones por varianza en tu corpus. (3) La curva naranja (sin renormalizar) es la del error típico: puede quedar por debajo de la azul porque el producto punto mezcla normas. Prueba a cambiar `emb` por un modelo con MRL y repite.

## 5 · Cuantización de embeddings: `int8` y binaria (desde cero)

Después de reducir las *dimensiones*, se reducen los *bits por dimensión*. Para $N$ documentos de $d$ dimensiones:

| Precisión | Bytes por vector | Con $d=1024$ | Ratio |
|---|---|---|---|
| `float32` | $4d$ | 4096 B | 1× |
| `float16` | $2d$ | 2048 B | 2× |
| `int8` | $d$ | 1024 B | 4× |
| binaria (1 bit) | $d/8$ | 128 B | 32× |

**Escalar `int8`.** Para cada dimensión $j$ se calibra un rango $[lo_j, hi_j]$ sobre una muestra del **corpus** y se mapea linealmente a 256 niveles:
$$q_{ij}=\operatorname{round}\!\Big(\tfrac{x_{ij}-lo_j}{(hi_j-lo_j)/255}\Big)-128\quad\in[-128,127].$$

**Binaria.** $b_{ij}=\mathbb{1}[x_{ij}>0]$, empaquetado 8 bits por byte. La similitud se aproxima con la **distancia de Hamming** (XOR + popcount). Para vectores $\pm1$, $\langle a,b\rangle=d-2\,H(a,b)$: es decir, ordenar por Hamming ordena por producto punto de los signos.

**Rescoring (clave).** La binaria sola pierde calidad; la receta estándar es **recuperar $k\times$`over` candidatos con Hamming** y **reordenarlos con la query en `float32` contra los vectores `int8`** (que pueden vivir en disco). Es la receta del post de Hugging Face (Shakir, Aarsen, Lee, 2024).

> ⚠️ **Calibra con el corpus, no con las queries**: las queries tienen una distribución distinta (más cortas) y el rango resultante es inadecuado. `sentence-transformers` recomienda calibrar con un conjunto grande de embeddings representativo de lo que vas a cuantizar (`calibration_embeddings`).

In [ ]:
POPCNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def ajusta_rangos(E_calib):
    """Rango [lo, hi] por dimensión calculado sobre el CORPUS (no sobre las queries)."""
    return E_calib.min(0), E_calib.max(0)

def cuantiza_int8(E, lo, hi):
    paso = np.maximum(hi - lo, 1e-9) / 255.0
    return np.clip(np.round((E - lo) / paso) - 128, -128, 127).astype(np.int8)

def descuantiza_int8(Q, lo, hi):
    paso = np.maximum(hi - lo, 1e-9) / 255.0
    return (Q.astype(np.float32) + 128) * paso + lo

def cuantiza_binario(E, umbral=0.0):
    return np.packbits(E > umbral, axis=1)                      # (N, d/8) uint8

def hamming(qb, db, bloque=2048):
    out = np.empty((len(qb), len(db)), dtype=np.int32)
    for s in range(0, len(db), bloque):
        x = qb[:, None, :] ^ db[None, s:s + bloque, :]            # XOR byte a byte
        out[:, s:s + bloque] = POPCNT[x].sum(-1)                  # popcount por tabla de 256 entradas
    return out

def topk_scores(S, k):
    idx = np.argpartition(-S, min(k, S.shape[1] - 1), axis=1)[:, :k]
    orden = np.take_along_axis(S, idx, 1).argsort(1)[:, ::-1]
    return np.take_along_axis(idx, orden, 1)

def busca_binario_rescore(Eq, Db, Qi8, lo, hi, k=10, over=4, umbral=0.0):
    H = hamming(np.packbits(Eq > umbral, axis=1), Db)
    cand = topk_scores(-H.astype(np.float32), min(k * over, H.shape[1]))     # menor Hamming = mejor
    out = np.empty((len(Eq), k), dtype=np.int64)
    for i in range(len(Eq)):
        v = descuantiza_int8(Qi8[cand[i]], lo, hi)
        out[i] = cand[i][np.argsort(-(v @ Eq[i]))[:k]]
    return out

# Verificación frente a sentence-transformers
try:
    from sentence_transformers.util import quantize_embeddings
except ImportError:
    from sentence_transformers.quantization import quantize_embeddings
lo, hi = ajusta_rangos(E_d)
mi_bin, st_bin = cuantiza_binario(E_d), quantize_embeddings(E_d, "ubinary")
mi_i8, st_i8 = cuantiza_int8(E_d, lo, hi), quantize_embeddings(E_d, "int8", ranges=np.stack([lo, hi]))
print("binario idéntico a sentence-transformers:", bool((mi_bin == st_bin).all()))
print("int8 difiere como máximo en", int(np.abs(mi_i8.astype(int) - st_i8.astype(int)).max()), "nivel (redondeo vs truncado)")
print(f"memoria del corpus: float32 {E_d.nbytes/1e3:.1f} KB · int8 {mi_i8.nbytes/1e3:.1f} KB · binario {mi_bin.nbytes/1e3:.1f} KB")

### Corpus ampliado para ver las diferencias
Con ≈ 90 documentos todo satisface al 100 %. Para estresar la cuantización **ampliamos el corpus con copias perturbadas** (casi-duplicados que actúan como *hard negatives*) y medimos qué fracción del **top-10 exacto en `float32`** conserva cada método (*neighbor recall@10*). Es la métrica estándar de ANN y aísla el error de cuantización del error del modelo.

In [ ]:
def aumenta(E, copias=40, sigma=0.5, seed=0):
    """Añade `copias` versiones ruidosas de cada vector (cos ≈ 0.9) para crear vecinos difíciles."""
    rng = np.random.default_rng(seed)
    bloques = [E]
    for _ in range(copias):
        ruido = rng.normal(size=E.shape)
        ruido *= sigma / np.linalg.norm(ruido, axis=1, keepdims=True)
        X = E + ruido
        bloques.append((X / np.linalg.norm(X, axis=1, keepdims=True)).astype(np.float32))
    return np.vstack(bloques)

def overlap_at_k(est, ref):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(est, ref)]))

def exacto_topk(Eq, Ed, k=10):
    return topk_scores(Eq @ Ed.T, k)

E_big = aumenta(E_d)
Dd = E_big.shape[1]
ref = exacto_topk(E_q, E_big)
lo, hi = ajusta_rangos(E_big)
Q8 = cuantiza_int8(E_big, lo, hi)
filas = []   # (método, bytes por vector en RAM, overlap@10)
filas.append(("float32", 4 * Dd, 1.0))
for frac in (2, 4, 8):
    m = max(Dd // frac, 8)
    filas.append((f"float32 · {m} dims", 4 * m, overlap_at_k(exacto_topk(trunc_renorm(E_q, m), trunc_renorm(E_big, m)), ref)))
S8 = E_q @ descuantiza_int8(Q8, lo, hi).T
filas.append(("int8", Dd, overlap_at_k(topk_scores(S8, 10), ref)))
Bb = cuantiza_binario(E_big)
filas.append(("binario (Hamming)", Dd // 8, overlap_at_k(topk_scores(-hamming(np.packbits(E_q > 0, axis=1), Bb).astype(np.float32), 10), ref)))
filas.append(("binario + rescoring int8 (x4)", Dd // 8, overlap_at_k(busca_binario_rescore(E_q, Bb, Q8, lo, hi, 10, 4), ref)))
mu = E_big.mean(0)
Bc = cuantiza_binario(E_big, mu)
filas.append(("binario centrado + rescoring", Dd // 8, overlap_at_k(busca_binario_rescore(E_q, Bc, Q8, lo, hi, 10, 4, umbral=mu), ref)))
import pandas as pd
tabla_q = pd.DataFrame(filas, columns=["método", "bytes/vec (RAM)", "overlap@10 vs float32"])
tabla_q["MB para 10 M docs"] = (tabla_q["bytes/vec (RAM)"] * 10_000_000 / 1e6).round(0)
display(tabla_q.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4))
for nombre, b, o in filas:
    color = "#1F6FEB" if "float32" in nombre else ("#2CA02C" if nombre == "int8" else "#F28E2B")
    ax.scatter(b, o, s=70, color=color, zorder=3)
    ax.annotate(nombre, (b, o), textcoords="offset points", xytext=(6, -10), fontsize=8)
ax.set_xscale("log", base=2); ax.set_xlabel("bytes por vector en RAM (escala log)"); ax.set_ylabel("overlap@10 con el top-10 float32")
ax.set_title("Calidad vs memoria: recortar dimensiones vs recortar bits"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) `int8` ocupa 4× menos y queda casi encima de `float32`: es la optimización "gratis". (2) La binaria sin rescoring cae mucho y **con rescoring** sube de forma notable ocupando 32× menos en RAM (los vectores `int8` pueden vivir en disco/SSD). *Cuánto* sube depende de la **dimensión**: con los 64 bits del fallback verás una recuperación parcial, mientras que con embeddings reales de 384–1024 dimensiones la binaria conserva mucha más información (el post de Hugging Face de 2024 reporta hasta ≈ 96 % del rendimiento con rescoring; mídelo con tu modelo). (3) Con el mismo presupuesto de bytes, **recortar bits suele dominar a recortar dimensiones** si el modelo no es Matryoshka; si lo es, combinarlos (p. ej. 512 dims binarias) forma la frontera de Pareto. (4) Centrar antes de binarizar ayuda cuando el modelo no produce coordenadas simétricas alrededor de 0. (5) **Memoria ≠ latencia**: en NumPy puro, Hamming con tabla de popcount es *más lento* que el producto matricial en `float32` (que usa BLAS). La binaria brilla en RAM; la velocidad llega con instrucciones `POPCNT`/SIMD de FAISS (`IndexBinaryFlat`) o de un motor de búsqueda, no con este código didáctico.

In [ ]:
factores = [1, 2, 4, 8, 16, 32]
ov = [overlap_at_k(busca_binario_rescore(E_q, Bb, Q8, lo, hi, 10, f), ref) for f in factores]
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(factores, ov, "o-", color="#F28E2B")
ax.set_xscale("log", base=2); ax.set_xticks(factores); ax.set_xticklabels(factores)
ax.set_xlabel("factor de sobre-muestreo 'over' (candidatos = 10 x over)"); ax.set_ylabel("overlap@10 con float32")
ax.set_title("Binaria + rescoring: más candidatos, más calidad (y más lecturas del int8)"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print({f: round(v, 3) for f, v in zip(factores, ov)})

> 👀 **Qué debes observar.** La curva sube rápido y luego se aplana: el rescoring de 4–10× candidatos suele bastar. Cada candidato extra implica **una lectura del vector `int8`** (a menudo de disco), así que `over` es un dial de latencia, no solo de calidad.

> ### 🎨 GEMINI-IMAGEN · ID: `M12-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **De float32 a int8 y a bits**
>
> **Qué muestra:** Tres representaciones de un mismo embedding (float32, int8, binario) con su coste en bytes y el flujo de búsqueda binaria + rescoring.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Un vector en float32, int8 y bits con su coste en bytes, y el flujo de búsqueda binaria con rescoring
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif. Sin elementos decorativos.
> 
> PARTE SUPERIOR (3 filas, de arriba abajo): un mismo vector de 8 coordenadas de ejemplo [0.12, -0.50, 0.33, 0.01, -0.27, 0.64, -0.08, 0.41].
> Fila 1 (gris): "float32 · 4 bytes por dimensión" mostrando los 8 números en cajas anchas. A la derecha: "d = 1024 → 4096 bytes".
> Fila 2 (azul): "int8 · 1 byte por dimensión" mostrando 8 enteros entre -128 y 127: [-22, -128, 38, -64, -91, 127, -75, 67]. A la derecha: "d = 1024 → 1024 bytes".
> Fila 3 (naranja): "binario · 1 bit por dimensión" mostrando 8 cajitas con los bits 1 0 1 1 0 1 0 1 (1 si la coordenada es positiva). A la derecha: "d = 1024 → 128 bytes".
> 
> PARTE INFERIOR: un flujo de izquierda a derecha con 3 cajas y flechas: "Query float32" → "Hamming (XOR + popcount) sobre bits: top 40 candidatos" (naranja) → "Rescoring: query float32 × vectores int8: top 10" (verde). Debajo una nota gris: "RAM: bits · Disco/SSD: int8".
> 
> Evita: valores numéricos distintos a los indicados, texto ilegible, flechas cruzadas, ilustraciones de chips, logos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M12-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Un vector en float32, int8 y bits con su coste en bytes, y el flujo de búsqueda binaria con rescoring](assets/M12-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> float32 : [ 0.12 -0.50  0.33  0.01 -0.27  0.64 -0.08  0.41]   4096 B (d=1024)
> int8    : [ -22 -128   38  -64   -91  127   -75   67]          1024 B
> bits    : [   1    0    1    1     0    1     0    1]           128 B
> flujo   : query f32 -> Hamming top-40 -> rescoring int8 -> top-10
> ```

> 👀 **Qué debes observar.** Cada fila reduce el tamaño: 4× (float32→int8) y luego 8× (int8→bit), en total 32×. El flujo inferior es la receta que usan los sistemas de búsqueda: la compresión fuerte filtra y la precisión alta decide.

## 6 · Con librerías de industria: comparar embedders reales

`sentence-transformers` unifica la carga y el `encode` de casi todas las familias (con `trust_remote_code=True` en las que traen código propio, como GTE y Nomic). Comparamos los candidatos del catálogo en **tres ejes**: calidad (nDCG@10 en Políglota), **memoria** (bytes del corpus por dimensión) y **latencia** (ms por documento/consulta en tu hardware).

- Con `FAST_DEV_RUN = True` solo se prueban 2 modelos pequeños. Con `False` y una GPU se intentan todos (los modelos con licencia, como EmbeddingGemma, requieren aceptar los términos en Hugging Face y un token `HF_TOKEN`; si fallan, se omiten).
- Si ningún modelo se puede descargar, la tabla usa **dos fallbacks TF-IDF+SVD** de distinta dimensión: no representan a ningún modelo real, pero mantienen el notebook ejecutable.

In [ ]:
import gc
CANDIDATOS = list(MODELOS) if (not FAST_DEV_RUN and device == "cuda") else ["multilingual-e5-small", "paraphrase-multilingual-MiniLM-L12-v2"]
filas_zoo, POR_MODELO = [], {}

def mide(clave, e):
    t0 = time.perf_counter(); Ed = e.encode_docs(DOC_TXT); t_doc = (time.perf_counter() - t0) / len(DOC_TXT) * 1e3
    t0 = time.perf_counter(); Eq = e.encode_queries([q["text"] for q in QRYS]); t_q = (time.perf_counter() - t0) / len(QRYS) * 1e3
    run = run_denso(Eq, Ed)
    res, pq = evaluate(run, QRELS), ndcgs_por_query(run, QRELS)
    POR_MODELO[clave] = dict(Eq=Eq, Ed=Ed, ndcg_q=pq)
    filas_zoo.append(dict(modelo=clave, real=e.is_real, dim=Ed.shape[1], params_M=round(e.n_params / 1e6, 1),
                          ndcg10=round(res["ndcg"], 3), recall10=round(res["recall"], 3),
                          ms_por_doc=round(t_doc, 2), ms_por_query=round(t_q, 2),
                          GB_10M_docs_f32=round(Ed.shape[1] * 4 * 10e6 / 1e9, 1)))

for clave in CANDIDATOS:
    e = cargar_embedder(clave)
    if e is not None:
        mide(clave, e)
    del e; gc.collect()
if len(filas_zoo) < 2:                                   # sin modelos reales: dos fallbacks para poder comparar
    for d_fb in (32, 64):
        mide(f"tfidf-svd-{d_fb} (fallback)", Embedder("fb", {}, fallback=HashSVDEmbedder(d_fb).fit(DOC_TXT)))
zoo = pd.DataFrame(filas_zoo).sort_values("ndcg10", ascending=False).reset_index(drop=True)
display(zoo)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4))
for r in zoo.itertuples():
    ax.scatter(r.ms_por_doc, r.ndcg10, s=40 + r.dim / 4, alpha=.75, color="#1F6FEB" if r.real else "#6B7280")
    ax.annotate(f"{r.modelo[:22]} ({r.dim}d)", (r.ms_por_doc, r.ndcg10), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("latencia de indexación (ms por documento, escala log)"); ax.set_ylabel("nDCG@10")
ax.set_title(f"Calidad vs latencia (tamaño = dimensión) · corpus {CORPUS['source']} · device={device}"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Busca la **frontera**: el punto arriba-izquierda domina. Un modelo 2–3× más lento solo vale la pena si sube varios puntos de nDCG **en tu corpus**. El tamaño de la burbuja (dimensión) es el coste *de almacenamiento*; la abscisa, el coste *de indexación*. Los puntos grises son fallbacks sin significado de modelo real. Recuerda que la latencia depende de GPU/CPU, `batch_size` y longitud máxima: repite la medición en tu hardware de producción.

### ¿La diferencia entre dos modelos es real? Bootstrap pareado
Con 58 queries, una diferencia de 0.02 de nDCG puede ser ruido. El **bootstrap pareado** remuestrea queries con reemplazo y calcula la distribución de la diferencia de medias; si el intervalo de confianza (IC) del 95 % incluye 0, no puedes afirmar que un modelo gane. (El módulo 15 formaliza esto.)

In [ ]:
def bootstrap_medias(x, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n, len(x)))
    return x[idx].mean(1)

def bootstrap_diff(a, b, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    d = (a[idx] - b[idx]).mean(1)
    return d.mean(), np.percentile(d, [2.5, 97.5]), float((d > 0).mean())

claves = list(POR_MODELO)
fig, ax = plt.subplots(figsize=(7, 0.7 * len(claves) + 1.5))
for i, c in enumerate(claves):
    medias = bootstrap_medias(POR_MODELO[c]["ndcg_q"])
    lo_, hi_ = np.percentile(medias, [2.5, 97.5])
    ax.errorbar(medias.mean(), i, xerr=[[medias.mean() - lo_], [hi_ - medias.mean()]], fmt="o", color="#1F6FEB", capsize=4)
ax.set_yticks(range(len(claves))); ax.set_yticklabels([c[:34] for c in claves]); ax.invert_yaxis()
ax.set_xlabel("nDCG@10 con IC 95 % (bootstrap sobre queries)"); ax.set_title("¿Se distinguen los modelos?"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
if len(claves) >= 2:
    a, b = claves[0], claves[1]
    media, ic, p = bootstrap_diff(POR_MODELO[a]["ndcg_q"], POR_MODELO[b]["ndcg_q"])
    print(f"{a} − {b}: Δ={media:+.3f} · IC95%=[{ic[0]:+.3f}, {ic[1]:+.3f}] · P(Δ>0)={p:.2f}")
    print("→ diferencia", "significativa" if ic[0] > 0 or ic[1] < 0 else "NO concluyente (el IC contiene 0)")

> 👀 **Qué debes observar.** Si las barras de error de dos modelos se solapan mucho y el IC de la diferencia contiene 0, **no hay evidencia para elegir uno**: decide por memoria, latencia o licencia. Con un golden set pequeño (decenas de queries) esto ocurre a menudo; la solución es **más queries**, no más modelos.

## 7 · MTEB y MMTEB: qué miden y qué no

**MTEB** (Muennighoff et al., 2022) agrupa tareas de 8 tipos (clasificación, clustering, clasificación de pares, reranking, retrieval, STS, resumen, bitext mining). **MMTEB** (Enevoldsen et al., 2025) lo extiende de forma colaborativa a cientos de tareas y a más de 250 idiomas. La librería `mteb` (v2 en 2026) expone `mteb.get_benchmark(...)`, `mteb.get_model(...)` y `mteb.evaluate(...)`.

**Límites que debes conocer**
1. Un **promedio** mezcla tareas que quizá no te importan (p. ej. clasificación de sentimiento) con tu caso (retrieval en tu dominio).
2. **Contaminación y entrenamiento en el benchmark:** hay modelos que ven conjuntos de entrenamiento de las propias tareas; la librería marca para ello un indicador de *zero-shot* (`show_zero_shot` en la definición de cada benchmark). Un modelo "primero" puede no generalizar a tus datos.
3. **Idioma y dominio:** pocas tareas miden documentación técnica ES/EN/código, que es lo que busca Políglota.
4. Las diferencias de décimas en el leaderboard suelen estar **dentro del ruido**.
5. Las cifras de ranking cambian cada semana: **no las copies en un documento de decisión sin fecha ni enlace**.

Veamos qué contiene realmente `MTEB(Multilingual, v2)` (esto no descarga datos; solo lee los metadatos de la librería).

In [ ]:
try:
    import mteb
    from collections import Counter
    bench = mteb.get_benchmark("MTEB(Multilingual, v2)")
    tipos = Counter(t.metadata.type for t in bench.tasks)
    idiomas = {l for t in bench.tasks for l in t.metadata.languages}
    idiomas_ret = {l for t in bench.tasks if t.metadata.type == "Retrieval" for l in t.metadata.languages}
    print(f"mteb {mteb.__version__} · {len(bench.tasks)} tareas · {len(idiomas)} códigos de idioma en total, "
          f"pero solo {len(idiomas_ret)} en las tareas de Retrieval · tipos: {dict(tipos)}")
    print("¿hay retrieval en español?", "spa" in idiomas_ret)
    fig, ax = plt.subplots(figsize=(7, 3.4))
    orden = sorted(tipos.items(), key=lambda kv: -kv[1])
    ax.barh([k for k, _ in orden], [v for _, v in orden], color="#1F6FEB"); ax.invert_yaxis()
    ax.set_xlabel("nº de tareas"); ax.set_title(f"Composición de MTEB(Multilingual, v2) · {len(bench.tasks)} tareas")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("mteb no disponible:", type(e).__name__, str(e)[:80])

> 👀 **Qué debes observar.** El benchmark es mayoritariamente de *clasificación, retrieval y STS* y los "cientos de idiomas" vienen sobre todo de tareas de *bitext mining*: si tu producto es retrieval, una parte importante del "promedio global" mide otra cosa y la cobertura de idiomas en retrieval es mucho menor. Por eso los equipos reportan **la columna de la tarea relevante** (p. ej. solo *Retrieval*), no el promedio.

Si tienes red, puedes **ejecutar una tarea pequeña** con `mteb` (celda protegida: cambia `RUN_MTEB = True`). `SciFact` es una de las tareas de retrieval más pequeñas de BEIR; el resultado principal es nDCG@10.

In [ ]:
RUN_MTEB = False          # pon True en Colab (necesita red y descarga el dataset + modelo)
if RUN_MTEB:
    import mteb
    modelo_mteb = mteb.get_model("intfloat/multilingual-e5-small")         # usa los prompts oficiales del modelo si MTEB los conoce
    tareas = mteb.get_tasks(tasks=["SciFact"])
    resultados = mteb.evaluate(modelo_mteb, tareas, encode_kwargs={"batch_size": 32})
    print(resultados)
else:
    print("RUN_MTEB=False: celda omitida (requiere red). La API v2 es mteb.get_model / mteb.get_tasks / mteb.evaluate.")

## 8 · Cómo elegir un embedder: árbol de decisión

```
¿Tus documentos pueden salir de tu infraestructura y aceptas coste por llamada?
├─ SÍ → API (OpenAI text-embedding-3 · Cohere Embed v4 · Voyage · Gemini Embedding)
│        └─ elige por: idioma (ES/EN/código), contexto, dimensiones Matryoshka, precio, residencia de datos
└─ NO → pesos abiertos
         ├─ ¿CPU / dispositivo / latencia muy baja?   → EmbeddingGemma-300M · multilingual-e5-small/base · gte-multilingual-base
         ├─ ¿Una GPU T4/L4 y calidad alta?              → Qwen3-Embedding-0.6B/4B · BGE-M3 · Nomic Embed v2 (MoE)
         ├─ ¿Necesitas sparse + multi-vector del mismo modelo? → BGE-M3 (módulo 13)
         └─ ¿Presupuesto de GPU grande y calidad máxima? → Qwen3-Embedding-8B · NV-Embed (revisa la licencia)

Después, SIEMPRE, en este orden:
 1) mide en TU golden set (con IC)        2) ¿cabe en RAM? → Matryoshka (dims)   3) → int8   4) → binario + rescoring
 5) ¿contexto largo o multimodal? → módulos 13 (late chunking) y 16        6) licencia y plan de migración (módulo 18)
```

La tabla siguiente resume **solo atributos que pude verificar** (a oct-2026; `—` = no verificado, consulta la tarjeta/documentación). Las cifras de dimensión, contexto y licencia **cambian con las versiones**.

In [ ]:
FAMILIAS = pd.DataFrame([
    # familia, acceso, dim. completa / Matryoshka, contexto, multimodal, notas
    ("multilingual-e5 (small/base/large)", "abierto (MIT)", "384 / 768 / 1024 · no", "512 (paper)", False, "prefijos query:/passage:; 94 idiomas según la tarjeta"),
    ("BGE-M3", "abierto (MIT)", "1024 / no", "8192", False, "denso + sparse + multi-vector en un modelo (Chen et al. 2024)"),
    ("gte-multilingual-base", "abierto (Apache-2.0)", "768 / dimensiones elásticas", "8192", False, "305 M parámetros; también genera vectores sparse"),
    ("nomic-embed-text-v2-moe", "abierto (Apache-2.0)", "768 → 256 / sí", "512", False, "MoE: 475 M totales, 305 M activos; ≈ 100 idiomas"),
    ("Qwen3-Embedding (0.6B/4B/8B)", "abierto (Apache-2.0)", "0.6B: hasta 1024 / sí (32–1024)", "32k", False, "decoder, last-token, instrucciones en la query"),
    ("EmbeddingGemma-300m", "pesos con licencia Gemma (gated)", "768 → 128 / sí", "2k", False, "308 M parámetros; prompts por tarea"),
    ("jina-embeddings-v5-text (nano/small)", "pesos; CC-BY-NC-4.0 según la tarjeta", "nano 768 · small 1024 / sí (hasta 32)", "small 32k; nano 8k (la tarjeta de nano dice 32k: verificar)", False, "feb-2026 (239 M / 677 M); LoRA por tarea; hay variante omni (may-2026)"),
    ("gemini-embedding-001 (API)", "API", "3072 / sí (128–3072)", "2048", False, "solo el vector de 3072 sale normalizado: renormaliza tú al reducir"),
    ("gemini-embedding-2 (API; preview 10-mar-2026, GA 22-abr-2026)", "API", "3072 / sí (128–3072)", "8192 (texto)", True, "texto, imagen, vídeo, audio y PDF en un espacio (módulo 16)"),
    ("OpenAI text-embedding-3-small/large (API)", "API", "1536 / 3072 · parámetro dimensions", "—", False, "ene-2024; el parámetro recorta dimensiones"),
    ("Cohere Embed v4 (API)", "API", "256/512/1024/1536 / sí", "128k", True, "texto + imágenes en un embedding"),
    ("Voyage-4 (API)", "API (large: cerrado)", "2048/1024/512/256 / sí + QAT", "32k", False, "anunciada el 15-ene-2026 (large, base, lite y nano de pesos abiertos); salidas float/int8/binario"),
], columns=["familia", "acceso", "dim / Matryoshka", "contexto", "multimodal", "notas"])

def shortlist(abierto=None, multimodal=None, matryoshka=None):
    t = FAMILIAS.copy()
    if abierto is not None:
        t = t[t["acceso"].str.contains("abierto|pesos", case=False) == abierto]
    if multimodal is not None:
        t = t[t["multimodal"] == multimodal]
    if matryoshka:
        t = t[t["dim / Matryoshka"].str.contains("sí|elásticas|parámetro", case=False)]
    return t[["familia", "acceso", "dim / Matryoshka", "contexto"]]

print("Caso Políglota (pesos abiertos + Matryoshka):")
display(shortlist(abierto=True, matryoshka=True))

> 👀 **Qué debes observar.** El filtro reduce 12 familias a unas pocas **candidatas**; el árbol no decide, **acota**. La decisión final sale de la tabla de la sección 6 (tu golden set) + el presupuesto de memoria de la sección 5.

### Ejemplo opcional: embeddings por API (protegido por variable de entorno)
No necesitas ninguna API para este curso. Si tienes `GEMINI_API_KEY`, esta celda embebe dos textos con `gemini-embedding-001`, reduce a 768 dimensiones y **renormaliza** (los vectores recortados no vienen unitarios).

In [ ]:
if os.environ.get("GEMINI_API_KEY"):
    from google import genai
    from google.genai import types
    cliente = genai.Client()
    r = cliente.models.embed_content(
        model="gemini-embedding-001",
        contents=["¿Cómo recorto un embedding?", "Truncate an embedding and renormalize it."],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768),
    )
    V = np.array([e.values for e in r.embeddings], dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True)            # renormalizar SIEMPRE tras reducir dimensiones
    print(V.shape, "coseno ES↔EN =", float(V[0] @ V[1]))
else:
    print("GEMINI_API_KEY no definida: celda omitida (opcional).")

## 9 · 🏭 En producción

- **OpenAI** ofrece `text-embedding-3-small` y `-large` (25-ene-2024) con el parámetro `dimensions`, que acorta el vector descartando valores del final conservando sus propiedades (documentación y anuncio de OpenAI). *Es Matryoshka de cara al usuario.*
- **Google** publicó `gemini-embedding-001` (paper: Lee et al., 2025) y **EmbeddingGemma** (≈ 300 M, pensado para ejecutarse en dispositivo; Google indica que cuantizado cabe en menos de 200 MB de RAM). **Gemini Embedding 2** salió en *vista previa pública* el 10-mar-2026 como primer modelo nativamente multimodal de Google y pasó a disponibilidad general (`gemini-embedding-2`) el 22-abr-2026 (blog de Google; módulo 16).
- **Cohere Embed v4**: dimensiones Matryoshka 256/512/1024/1536, contexto de 128k y entrada multimodal (changelog oficial de Cohere).
- **Voyage-4** (familia): según la documentación de Voyage en MongoDB/Azure, entrenada con Matryoshka y *quantization-aware training*, con salidas float/int8/binario y un espacio de embeddings compartido entre tamaños (pudiendo indexar con el modelo grande y consultar con uno más barato). La familia (voyage-4-large, voyage-4, voyage-4-lite y voyage-4-nano de pesos abiertos) se anunció el 15-ene-2026 (MongoDB); algunos catálogos de terceros muestran fechas posteriores de versión.
- **Jina** (v3 con *task LoRA*, v4 multimodal, v5 en feb-2026 y v5-omni en may-2026 según Elastic), **Alibaba** (Qwen3-Embedding, Apache-2.0), **Nomic** (código y datos abiertos) y **Hugging Face** (TEI para servir, módulo 18) cubren el mundo de pesos abiertos.
- **Elastic** y otros motores aplican cuantización binaria derivada de RaBitQ (BBQ) para reducir memoria (módulo 14).

**Aritmética de coste** (10 M documentos, $d=1024$): `float32` = 40.96 GB · `int8` = 10.24 GB · binario = 1.28 GB (+ 10.24 GB de `int8` en disco si haces rescoring). Con $d=256$ Matryoshka + `int8`: 2.56 GB. Esa es la diferencia entre un clúster y un nodo.

## 10 · 🧠 Secretos de la élite

1. **Tu golden set manda sobre MTEB.** La librería `mteb` marca explícitamente qué resultados son *zero-shot*; un modelo puede liderar el promedio por haber visto datos de las tareas. Valida con IC (bootstrap) en tus queries (módulo 15).
2. **El prefijo/instrucción no es opcional.** Las tarjetas de E5, Nomic, Qwen3-Embedding y EmbeddingGemma definen formatos distintos; se reporta (tarjeta de Qwen3-Embedding, citada en un export ONNX de Qdrant) que omitir la instrucción de la query cuesta del orden del 1–5 % en retrieval. Un fallo de prefijo es silencioso: ningún error, solo peor ranking.
3. **Last-token + padding a la derecha = bug silencioso.** Usa siempre el índice del último token *real* (`attention_mask.sum(1)-1`) o `padding_side="left"`; lo reproducimos en la sección 2.
4. **Calibra `int8` con el corpus, no con las queries.** `sentence-transformers` pide `calibration_embeddings` representativos; rangos calculados con pocas queries (más cortas, otra distribución) recortan los valores del corpus.
5. **Renormaliza siempre tras truncar o cuantizar.** No cuesta nada y evita mezclar normas en el producto punto. Con `gemini-embedding-001`, fuentes de terceros indican que solo la salida de 3072 dimensiones viene normalizada: verifica en la documentación y renormaliza.
6. **Binaria = barata en RAM, pero con rescoring.** El post de Hugging Face (2024) y la documentación de `sentence-transformers` reportan hasta ≈ 96 % del rendimiento con 32× menos memoria *con rescoring*; la cifra depende del modelo y de la dimensión: mídela.
7. **La dimensión es una decisión de capacidad, no solo de memoria.** Weller et al. (2025) muestran, teórica y empíricamente (dataset sintético LIMIT), que una dimensión fija limita qué conjuntos top-k pueden representarse: dimensiones muy pequeñas fallan en consultas "combinatorias" aunque el promedio de MTEB parezca bueno. Se retoma en el módulo 15.
8. **Cambiar de embedder = reindexar todo.** Planifica versionado y migración (módulo 18); un espacio compartido entre tamaños (reportado para Voyage-4) permite asimetría query/documento sin reindexar.

## 11 · ⚠️ Errores comunes

| Error | Síntoma | Arreglo |
|---|---|---|
| Olvidar `query:`/`passage:` (o la instrucción) | nDCG baja unos puntos sin error | centraliza el formateo en un único `Embedder` y testéalo |
| Truncar un modelo sin MRL | calidad se desploma a < 256 dims | usa un modelo Matryoshka o PCA |
| No renormalizar tras truncar | ranking inestable con producto punto | `e / ‖e‖` siempre |
| Calibrar `int8` con las queries | pérdida de calidad inexplicable | rangos sobre el corpus |
| Binarizar sin rescoring | recall muy inferior | `top-(k·over)` con Hamming + rescoring `int8` |
| Mezclar `padding_side` | embeddings de decoders raros | índice del último token real |
| Comparar modelos con 20 queries sin IC | "ganador" que cambia al re-muestrear | bootstrap pareado y más queries |
| Cortar `max_seq_length` sin mirar la fertilidad | español truncado antes que inglés | mide longitud en tokens por idioma |
| Mezclar embeddings de dos modelos/versiones en el mismo índice | recall aleatorio | un índice por (modelo, versión, dimensión, precisión) |
| Citar un rango de leaderboard sin fecha | decisión basada en dato caducado | fecha + enlace en el documento de decisión |

## 12 · Autoevaluación

<details><summary><b>1 · (Cálculo)</b> Tienes 25 M de documentos con $d=768$. ¿Cuánta RAM ocupa el índice en <code>float32</code>, en <code>int8</code> y en binario? ¿Y si usas Matryoshka a 256 dimensiones + <code>int8</code>?</summary>

`float32`: $25\cdot10^6\cdot768\cdot4 = 76.8$ GB. `int8`: $19.2$ GB. Binario: $768/8=96$ B por vector → $2.4$ GB. Matryoshka 256 + `int8`: $256$ B por vector → $6.4$ GB. (Más los metadatos y la estructura del índice ANN, que no están incluidos.)
</details>

<details><summary><b>2 · (Cálculo)</b> Dos vectores binarios de 1024 bits difieren en 100 bits. ¿Cuál es el producto punto de sus versiones $\pm1$ y su "coseno"?</summary>

$\langle a,b\rangle = d-2H = 1024-200 = 824$. Como $\lVert a\rVert=\lVert b\rVert=\sqrt{1024}$, el coseno es $824/1024\approx0.805$.
</details>

<details><summary><b>3 · (Diagnóstico)</b> Cambias de un modelo a otro de la misma familia y el nDCG@10 cae 6 puntos, sin errores. ¿Qué compruebas primero?</summary>

Los **prefijos/instrucciones** del nuevo modelo (cada versión puede cambiarlos), la **normalización** de los vectores, el **pooling/padding side** y el `max_seq_length`. Después, que el índice se haya reconstruido con el modelo nuevo (no mezclar espacios).
</details>

<details><summary><b>4 · (Diagnóstico)</b> Truncas un embedder de 768 a 128 dimensiones y el nDCG@10 pasa de 0.62 a 0.31. ¿Por qué y qué haces?</summary>

Muy probablemente el modelo **no** se entrenó con Matryoshka: la información está repartida por todas las coordenadas. Opciones: usar un modelo con MRL (Nomic v2, Qwen3-Embedding, EmbeddingGemma…), aprender una proyección **PCA** sobre tu corpus, o recortar bits (`int8`/binario) en lugar de dimensiones.
</details>

<details><summary><b>5 · (Transferencia)</b> Políglota tiene 50 M de fragmentos ES/EN/código y un nodo de 32 GB de RAM para el índice. Diseña la configuración de embedder, dimensión y precisión.</summary>

Presupuesto ≈ 20 GB para vectores (el resto, grafo ANN y overhead) → ≈ 400 B por vector con $50\cdot10^6$ vectores. Opciones: **384 dims `int8`** = 19.2 GB (justo), **256 dims `int8`** = 12.8 GB, o **512 dims binarias** = 3.2 GB en RAM con los `int8` de 512 dims (25.6 GB) en SSD para rescoring. Elegiría un modelo con MRL multilingüe que entienda código (Qwen3-Embedding-0.6B, Nomic v2 o EmbeddingGemma), mediría nDCG@10 en el golden set con IC para cada (dim, precisión) y escogería el punto de la frontera de Pareto que cumpla el SLA de latencia y recall.
</details>

<details><summary><b>6 · (Transferencia)</b> ¿Por qué un decoder con last-token pooling necesita que el texto termine con un token especial (EOS)?</summary>

Porque el vector es el estado del **último token**, y con atención causal solo ese estado ha atendido a todo el texto. El token final designado hace de "marcador de resumen" y el modelo se entrena para que ese estado condense el significado.
</details>

<details><summary><b>7 · (Recuerdo)</b> ¿Qué dos ingredientes de entrenamiento hacen que un prefijo del vector sea útil por sí solo?</summary>

La pérdida contrastiva evaluada en **varios prefijos** (anidados) y la **renormalización de cada prefijo** dentro de la pérdida.
</details>

## 13 · Referencias (verificadas; consultadas en oct-2026)

**Papers (arXiv)**
- Kusupati et al. (2022). *Matryoshka Representation Learning*. <https://arxiv.org/abs/2205.13147>
- Wang et al. (2024). *Multilingual E5 Text Embeddings: A Technical Report*. <https://arxiv.org/abs/2402.05672>
- Wang et al. (2023). *Improving Text Embeddings with Large Language Models* (e5-mistral). <https://arxiv.org/abs/2401.00368>
- Chen et al. (2024). *M3-Embedding: Multi-Linguality, Multi-Functionality, Multi-Granularity Text Embeddings Through Self-Knowledge Distillation* (BGE-M3). <https://arxiv.org/abs/2402.03216>
- Zhang et al. (2024). *mGTE: Generalized Long-Context Text Representation and Reranking Models for Multilingual Text Retrieval*. <https://arxiv.org/abs/2407.19669>
- Nussbaum et al. (2024). *Nomic Embed: Training a Reproducible Long Context Text Embedder*. <https://arxiv.org/abs/2402.01613> · Nussbaum & Duderstadt (2025). *Training Sparse Mixture Of Experts Text Embedding Models*. <https://arxiv.org/abs/2502.07972>
- Lee et al. (2024). *NV-Embed: Improved Techniques for Training LLMs as Generalist Embedding Models*. <https://arxiv.org/abs/2405.17428>
- BehnamGhader et al. (2024). *LLM2Vec: Large Language Models Are Secretly Powerful Text Encoders*. <https://arxiv.org/abs/2404.05961>
- Zhang et al. (2025). *Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models*. <https://arxiv.org/abs/2506.05176>
- Lee et al. (2025). *Gemini Embedding: Generalizable Embeddings from Gemini*. <https://arxiv.org/abs/2503.07891>
- Vera et al. (2025). *EmbeddingGemma: Powerful and Lightweight Text Representations*. <https://arxiv.org/abs/2509.20354>
- Sturua et al. (2024). *jina-embeddings-v3: Multilingual Embeddings With Task LoRA*. <https://arxiv.org/abs/2409.10173> · Günther et al. (2025). *jina-embeddings-v4: Universal Embeddings for Multimodal Multilingual Retrieval*. <https://arxiv.org/abs/2506.18902>
- Muennighoff et al. (2022). *MTEB: Massive Text Embedding Benchmark*. <https://arxiv.org/abs/2210.07316> · Enevoldsen et al. (2025). *MMTEB: Massive Multilingual Text Embedding Benchmark*. <https://arxiv.org/abs/2502.13595>
- Weller et al. (2025). *On the Theoretical Limitations of Embedding-Based Retrieval*. <https://arxiv.org/abs/2508.21038>
- Warner et al. (2024). *Smarter, Better, Faster, Longer: A Modern Bidirectional Encoder…* (ModernBERT). <https://arxiv.org/abs/2412.13663>

**Blogs de ingeniería y documentación**
- Shakir, Aarsen, Lee (2024). *Binary and Scalar Embedding Quantization for Significantly Faster & Cheaper Retrieval*. <https://huggingface.co/blog/embedding-quantization>
- Documentación de `sentence-transformers`: *Embedding Quantization*. <https://sbert.net/examples/applications/embedding-quantization/README.html>
- OpenAI (2024). *New embedding models and API updates*. <https://openai.com/index/new-embedding-models-and-api-updates/>
- Google (2026). *Gemini Embedding 2: Our first natively multimodal embedding model*. <https://blog.google/innovation-and-ai/models-and-research/gemini-models/gemini-embedding-2/>
- Google (22-abr-2026). *Gemini Embedding 2 is now generally available*. <https://blog.google/innovation-and-ai/models-and-research/gemini-models/gemini-embedding-2-generally-available/>
- Google AI for Developers. *EmbeddingGemma model overview*. <https://ai.google.dev/gemma/docs/embeddinggemma>
- Cohere. *Announcing Embed Multimodal v4* (changelog). <https://docs.cohere.com/v2/changelog/embed-multimodal-v4>
- MongoDB / Voyage AI. *Dimensions and quantization* (tutorial). <https://www.mongodb.com/docs/voyageai/tutorials/dimensions-and-quantization/>
- Jina AI. *jina-embeddings-v5-text-small*. <https://jina.ai/en-US/models/jina-embeddings-v5-text-small/>
- Isaac Chung (blog de Hugging Face). *Introducing MTEB v2*. <https://huggingface.co/blog/isaacchung/mteb-v2> · paquete `mteb` en PyPI. <https://pypi.org/project/mteb/>

**Para profundizar:** la lección del módulo 13 (BGE-M3 en sus tres modos), el módulo 14 (qué hace un índice ANN con estos vectores y RaBitQ) y el módulo 15 (evaluación rigurosa).